<a href="https://colab.research.google.com/github/armandoordonez/AI-Engineering/blob/main/function_calling_tool_use_gemini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1 · Function Calling / Tool Use (Gemini)
### Sesión 1 — Agentes de IA, Orquestación y Protocolos

**Complejidad: 🟢 Básica**   |   **Dependencias: solo `google-genai`**

Este es el notebook más simple de la serie. Usamos la **Interactions API** de Gemini (`client.interactions.create`), que es la interfaz vigente recomendada por Google para tool use y agentes desde 2026.

**Qué vas a construir:** el ciclo básico `request → function_call → function_result → respuesta final`, con **una sola vuelta** (en el notebook 2 lo convertimos en un loop completo).

> Modelo usado en este notebook: `gemini-3.5-flash`. Puedes cambiarlo por cualquier otro modelo Gemini vigente editando la constante `MODEL`.


## 🎯 Objetivo de aprendizaje

Al terminar este notebook vas a poder:
- Explicar por qué un LLM no puede ejecutar acciones por sí solo, y qué problema resuelve *function calling / tool use*.
- Definir el schema de una herramienta (`name`, `description`, `parameters`) para que el modelo la invoque correctamente.
- Implementar el ciclo completo de una sola vuelta: enviar la herramienta → recibir la solicitud del modelo → ejecutar la función real → devolver el resultado → obtener la respuesta final.


## 📚 Teoría: Function Calling / Tool Use

Un LLM, por sí solo, **solo genera texto**. No puede consultar una base de datos, llamar a una API, ni saber si el clima de hoy cambió — su conocimiento es estático y no tiene manos para actuar en el mundo real.

*Function calling* (también llamado *tool use*) es el mecanismo que resuelve esto: tú le describes al modelo, mediante un **schema estructurado**, qué funciones existen y qué parámetros reciben. El modelo **decide** cuándo necesita usar una, y en vez de responder texto libre, responde una estructura indicando qué función llamar y con qué argumentos. Nunca ejecuta la función él mismo — eso siempre es responsabilidad de tu código.

El ciclo tiene 4 pasos:
1. Tu app envía el prompt + la definición de herramientas al modelo.
2. El modelo responde pidiendo ejecutar una función (o responde texto normal si no la necesita).
3. Tu código ejecuta la función real.
4. Envías el resultado de vuelta al modelo, que genera la respuesta final para el usuario.

**El detalle que más importa:** la calidad de la `description` de cada herramienta es lo que determina si el modelo la usa en el momento correcto y con los parámetros correctos — es, en la práctica, el "manual de instrucciones" que el modelo lee para decidir.


## 0. Instalación (única dependencia)

In [1]:
!pip install -q google-genai

### Configurar API key de Gemini

**Cómo obtenerla:** [aistudio.google.com/apikey](https://aistudio.google.com/apikey) (gratis, dos clics).

Recomendado en Colab: guárdala en **Secrets** (ícono de llave 🔑 a la izquierda) con el nombre `GEMINI_API_KEY` y actívala para este notebook. Si no usas Secrets, te la pedirá por input.

⚠️ **Aviso conocido (2026):** Google está migrando las API keys al nuevo formato con prefijo `AQ.` (antes `AIza...`). Hay reportes activos y aún no resueltos en el foro oficial de Google de que las keys `AQ.` devuelven `401 ACCESS_TOKEN_TYPE_UNSUPPORTED` en algunas cuentas/proyectos, incluso bien configuradas. La celda de abajo te dice qué tipo de key tienes para descartar esto como causa del error.


In [2]:
import os

try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
except Exception:
    from getpass import getpass
    os.environ["GEMINI_API_KEY"] = os.environ.get("GEMINI_API_KEY") or getpass("Pega tu GEMINI_API_KEY: ")

_key = os.environ.get("GEMINI_API_KEY", "")
print("API key configurada:", "OK" if _key else "FALTA")

if _key.startswith("AQ."):
    print("ADVERTENCIA: tu key tiene el nuevo formato 'AQ.' -- si mas adelante ves un error 401")
    print("ACCESS_TOKEN_TYPE_UNSUPPORTED, es un problema conocido y actualmente activo del lado de")
    print("Google con este formato de key, no de este notebook. Revisa:")
    print("https://discuss.ai.google.dev/c/gemini-api/4  (buscar 'AQ. 401 ACCESS_TOKEN_TYPE_UNSUPPORTED')")
elif _key.startswith("AIza"):
    print("Formato de key clasico (AIza...) -- no deberia verse afectado por el problema de las keys 'AQ.'.")


API key configurada: OK
ADVERTENCIA: tu key tiene el nuevo formato 'AQ.' -- si mas adelante ves un error 401
ACCESS_TOKEN_TYPE_UNSUPPORTED, es un problema conocido y actualmente activo del lado de
Google con este formato de key, no de este notebook. Revisa:
https://discuss.ai.google.dev/c/gemini-api/4  (buscar 'AQ. 401 ACCESS_TOKEN_TYPE_UNSUPPORTED')


## 1. Definir una herramienta (function declaration)

Una herramienta se define con: `type: "function"`, `name`, `description` (crítico: el modelo decide **cuándo** usarla en base a esto) y `parameters` (JSON Schema).


In [3]:
from google import genai

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])  # explícito: evita depender de la autodetección de entorno

MODEL = "gemini-3.5-flash"

# Herramienta simple: consultar el clima de una ciudad (simulado)
weather_function = {
    "type": "function",
    "name": "get_weather",
    "description": "Obtiene el clima actual de una ciudad. Úsala cuando el usuario pregunte por el clima o temperatura de un lugar.",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {"type": "string", "description": "Nombre de la ciudad, ej: Cali, Bogotá"}
        },
        "required": ["city"]
    }
}

# Implementación real de la herramienta (aquí simulada; en producción llamarías a una API real)
def get_weather(city: str) -> str:
    fake_db = {"cali": "28°C, soleado", "bogota": "16°C, nublado", "medellin": "23°C, parcialmente nublado"}
    return fake_db.get(city.lower(), f"No tengo datos de clima para {city}")


## 2. Primer turno: el modelo decide si necesita la herramienta

Le enviamos un mensaje que **requiere** usar la herramienta. En vez de responder texto, el modelo genera un `step` de tipo `function_call`.


In [4]:
interaction = client.interactions.create(
    model=MODEL,
    input="¿Qué clima hace hoy en Cali?",
    tools=[weather_function],
)

for step in interaction.steps:
    print(step.type, "->", step)


function_call -> arguments={'city': 'Cali'} id='0ga7jex0' name='get_weather' type='function_call' signature='EqgCCqUCARFNMg8tt6QskXZaSq2AA7gljWQEnavDkrM+bF6WrqRq4VzA8f6XL8IevWQ1chR9rK3P3NG0piexrkc6YUvo2fWqKcXhbmZL2+0pY08havg7fqc9DOIQa/ugyMM4yIi2OGUamzsFYIXBNEky8J+sm+77hFPICyK2QM6JpPuHwV4HJ4y1Ssh520KKVVG+C4AQh41WAsFuL8PVxV3cpBiILLY8AzkWt1JhRq/sE0SLgJ0X5lJKLh49rovJTygjGDmN26jDh98zIvOn26NN4W1zLOKlJO5+htkJtF39UFcxtNk86EzUlKgbQ7Q0KOTENafbDwdmogQFMehc1K8TWE19aIz9Gjs1D0F45ldHOYJkR3vkE9WvLUL23MxWYc51m2Zd8yQV6EI='


## 3. Ejecutar la herramienta y devolver el resultado

Buscamos el `step` de tipo `function_call`, ejecutamos la función real, y enviamos el resultado de vuelta con `function_result`, encadenando con `previous_interaction_id`.


In [5]:
fc_step = next(s for s in interaction.steps if s.type == "function_call")
print(f"Función a llamar: {fc_step.name}({fc_step.arguments})")

if fc_step.name == "get_weather":
    result = get_weather(**fc_step.arguments)

final_interaction = client.interactions.create(
    model=MODEL,
    input=[{
        "type": "function_result",
        "name": fc_step.name,
        "call_id": fc_step.id,
        "result": [{"type": "text", "text": result}],
    }],
    tools=[weather_function],
    previous_interaction_id=interaction.id,
)

print(final_interaction.output_text)


Función a llamar: get_weather({'city': 'Cali'})
Hoy en Cali hace un clima de **28°C y está soleado**. ¡Un día perfecto para disfrutar del calor de la sucursal del cielo!


## 🧪 Ejercicio

Agrega una segunda herramienta `convert_currency(amount, from_currency, to_currency)` (puedes simular tasas de cambio fijas) y haz una pregunta que combine ambas herramientas, por ejemplo: *"¿Qué clima hace en Bogotá y cuánto son 50 USD en COP?"*. Observa cuántos `steps` de tipo `function_call` genera el modelo en un mismo turno (llamado *parallel function calling*).

---
**Siguiente notebook:** `agente_react_manual.ipynb` — convertimos esto en un loop completo (el ciclo ReAct de la charla).


In [6]:
# Tu código aquí
